# 04. 일기 감정 분석 시스템 (12주차)

**흐름**: 일기 → 문장 분리 → 문장별 44감정 예측(최종 모델) → 하루 감정 비율 → 날짜별 저장 → 주간/월간 그래프 → 감정 캘린더 → 코멘트

| 단계 | 방법 |
|---|---|
| 문장 분리 | 마침표·줄바꿈 + **연결어미(~지만, ~는데 등)**에서도 끊음 → 한 문장에 섞인 감정 분리 |
| 문장별 감정 | `kobert_kote_nall_e5.pt` (5에폭 + pos_weight), 저장된 임계값(0.6) 이상인 감정 선택 |
| **하루 감정 비율** | 감정이 선택된 문장 수 ÷ 전체 문장 수 (단순 확률 평균은 한 문장에서만 강한 감정이 묻혀서 안 씀) |
| 감정 그룹 | 44개 → **7개 그룹**(기쁨·만족 / 기대·관심 / 슬픔·실망 / 분노·불만 / 불안·당황 / 피로·지루함 / 없음) |
| 긍정·부정 점수 | 문장마다 (긍정 감정 최대 확률 − 부정 감정 최대 확률) → 하루 평균, -1 ~ +1 |
| 저장 | `outputs/diary_daily.csv`(하루 단위), `outputs/diary_sentences.csv`(문장 단위), 그래프 png |

**실행**: 커널 `kobert` 선택 → **Run All**. (학습 없음, 모델 불러와서 예측만 → 1~2분)
**내 일기 넣기**: `data/sample_diary.csv`에 `date,text` 줄을 추가하거나, 맨 아래 `write_diary()` 셀 사용.

In [1]:
# 1. 기본 설정
import re, calendar
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle, Patch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

project_root = Path.cwd().parent
model_dir  = project_root / "models"
data_dir   = project_root / "data"
output_dir = project_root / "outputs"
output_dir.mkdir(exist_ok=True)

# 한글 폰트 (윈도우: 맑은 고딕)
for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f
        break
plt.rcParams["axes.unicode_minus"] = False
print("font:", plt.rcParams["font.family"])

device: cuda
font: ['Malgun Gothic']


In [2]:
# 2. 설정 (여기만 바꾸면 됨)
MODEL_FILE        = "kobert_kote_nall_e5.pt"    # 최종 모델
DIARY_FILE        = data_dir / "sample_diary.csv" # 일기 파일 (date,text)
THRESHOLD         = None    # None 이면 모델에 저장된 값(0.6) 사용
SPLIT_CONNECTIVES = True    # 연결어미(~지만, ~는데 ...)에서도 문장 나누기

In [3]:
# 3. 감정 이름 + 7개 그룹
LABELS = ['불평/불만','환영/호의','감동/감탄','지긋지긋','고마움','슬픔','화남/분노','존경','기대감','우쭐댐/무시함',
          '안타까움/실망','비장함','의심/불신','뿌듯함','편안/쾌적','신기함/관심','아껴주는','부끄러움','공포/무서움','절망',
          '한심함','역겨움/징그러움','짜증','어이없음','없음','패배/자기혐오','귀찮음','힘듦/지침','즐거움/신남','깨달음',
          '죄책감','증오/혐오','흐뭇함(귀여움/예쁨)','당황/난처','경악','부담/안 내킴','서러움','재미없음','불쌍함/연민','놀람',
          '행복','불안/걱정','기쁨','안심/신뢰']

GROUPS = {
    "기쁨·만족": ['환영/호의','감동/감탄','고마움','존경','뿌듯함','편안/쾌적','아껴주는','흐뭇함(귀여움/예쁨)',
                 '행복','기쁨','즐거움/신남','안심/신뢰'],
    "기대·관심": ['기대감','신기함/관심','깨달음','비장함'],
    "슬픔·실망": ['슬픔','안타까움/실망','절망','서러움','불쌍함/연민','패배/자기혐오','죄책감','부끄러움'],
    "분노·불만": ['불평/불만','화남/분노','짜증','어이없음','한심함','역겨움/징그러움','증오/혐오','의심/불신','우쭐댐/무시함'],
    "불안·당황": ['공포/무서움','불안/걱정','당황/난처','경악','놀람','부담/안 내킴'],
    "피로·지루함": ['지긋지긋','귀찮음','힘듦/지침','재미없음'],
    "없음": ['없음'],
}
GROUP_COLORS = {"기쁨·만족": "#F2B134", "기대·관심": "#5DBB8F", "슬픔·실망": "#5B8DEF", "분노·불만": "#E5534B",
                "불안·당황": "#9B6FD3", "피로·지루함": "#A47551", "없음": "#BDBDBD"}
POSITIVE = {"기쁨·만족", "기대·관심"}
NEGATIVE = {"슬픔·실망", "분노·불만", "불안·당황", "피로·지루함"}

GROUP_NAMES = list(GROUPS)
GROUP_IDX = {g: [LABELS.index(e) for e in es] for g, es in GROUPS.items()}
POS_IDX = [i for g in POSITIVE for i in GROUP_IDX[g]]
NEG_IDX = [i for g in NEGATIVE for i in GROUP_IDX[g]]
assert sorted(sum(GROUP_IDX.values(), [])) == list(range(44)), "44개 감정이 그룹에 빠짐없이 한 번씩 들어가야 함"
print({g: len(v) for g, v in GROUP_IDX.items()})

{'기쁨·만족': 12, '기대·관심': 4, '슬픔·실망': 8, '분노·불만': 9, '불안·당황': 6, '피로·지루함': 4, '없음': 1}


In [4]:
# 4. 최종 모델 불러오기 (학습 X)
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel

MODEL_NAME = "skt/kobert-base-v1"
tokenizer = KoBERTTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.encode("한국어 모델을 공유합니다.") == [2, 4958, 6855, 2046, 7088, 1050, 7843, 54, 3], "토크나이저 이상!"

class KoBERTMultiLabelClassifier(nn.Module):
    def __init__(self, num_labels=44, dropout=0.2):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        return self.classifier(self.dropout(out.last_hidden_state[:, 0, :]))

ckpt = torch.load(model_dir / MODEL_FILE, map_location=device, weights_only=False)
model = KoBERTMultiLabelClassifier().to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()
TH = THRESHOLD if THRESHOLD is not None else ckpt.get("threshold", 0.5)
MAX_LENGTH = ckpt.get("max_length", 64)
print(f"모델: {MODEL_FILE} | 임계값: {TH} | max_length: {MAX_LENGTH}")

@torch.no_grad()
def predict_sentences(sents, batch_size=32):
    """문장 리스트 → (문장 수, 44) 확률"""
    out = []
    for i in range(0, len(sents), batch_size):
        enc = tokenizer(sents[i:i+batch_size], padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                       token_type_ids=enc.get("token_type_ids"))
        out.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 44))

D:\김민서\박상현_작업본_1031\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.
D:\김민서\박상현_작업본_1031\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\김민서\\박상현_작업본_1102\\models\\kobert_kote_nall_e5.pt'

In [ ]:
# 5. 문장 분리
CONNECTIVE = re.compile(r"(?<=지만|는데|은데|던데|인데|으나|면서)\s+")   # 연결어미 뒤 띄어쓰기에서 자르기

def split_sentences(text):
    sents = [s.strip() for s in re.split(r"(?<=[.!?~])\s+|\n+", str(text)) if s.strip()]
    if SPLIT_CONNECTIVES:
        sents = [p.strip() for s in sents for p in CONNECTIVE.split(s) if p.strip()]
    return sents

print(split_sentences("오늘 시험을 잘 봐서 뿌듯했지만 다음 발표가 걱정되고 너무 피곤하다. 그래도 내일은 쉰다!"))

In [ ]:
# 6. 하루 일기 분석 함수
def analyze_text(text):
    sents = split_sentences(text)
    probs = predict_sentences(sents)                     # (문장 수, 44)
    picked = probs >= TH                                 # 문장별 선택된 감정

    emo_ratio = picked.mean(0)                           # 감정별: 선택된 문장 비율
    group_ratio = {g: picked[:, idx].any(1).mean() for g, idx in GROUP_IDX.items()}   # 그룹별 문장 비율
    group_strength = {g: probs[:, idx].max(1).mean() for g, idx in GROUP_IDX.items()} # 동점일 때 비교용
    valence = float((probs[:, POS_IDX].max(1) - probs[:, NEG_IDX].max(1)).mean())     # -1(부정) ~ +1(긍정)

    cand = [g for g in GROUP_NAMES if g != "없음"]
    main = max(cand, key=lambda g: (group_ratio[g], group_strength[g]))
    if group_ratio[main] == 0:
        main = "없음"
    top = [LABELS[i] for i in emo_ratio.argsort()[::-1][:3] if emo_ratio[i] > 0]

    sent_rows = [{"sentence": s,
                  "emotions": ", ".join(f"{LABELS[i]} {probs[k, i]:.2f}" for i in probs[k].argsort()[::-1] if picked[k, i]) or "없음"}
                 for k, s in enumerate(sents)]
    day = {"n_sent": len(sents), "valence": round(valence, 3), "main_group": main,
           "top_emotions": ", ".join(top),
           **{f"r_{g}": round(float(group_ratio[g]), 3) for g in GROUP_NAMES}}
    return day, sent_rows, emo_ratio

# 지난번 예시: 확률 평균(기존) vs 문장 비율(새 방식)
ex = "오늘 시험을 잘 봐서 뿌듯했다. 그런데 다음 주 발표가 걱정된다. 알바까지 하고 와서 너무 피곤하다."
d, rows, er = analyze_text(ex)
for r in rows: print("-", r["sentence"], "\n    →", r["emotions"])
mean_p = predict_sentences(split_sentences(ex)).mean(0)
print("\n[기존: 확률 평균 Top5]", ", ".join(f"{LABELS[i]} {mean_p[i]:.2f}" for i in mean_p.argsort()[::-1][:5]))
print("[새 방식: 문장 비율 Top5]", ", ".join(f"{LABELS[i]} {er[i]:.0%}" for i in er.argsort()[::-1][:5]))
print("[그룹 비율]", ", ".join(f"{g} {d['r_'+g]:.0%}" for g in GROUP_NAMES if d['r_'+g] > 0))
print("[대표 그룹]", d["main_group"], "| 긍정·부정 점수", d["valence"])

In [ ]:
# 7. 일기 불러오기 (파일이 없으면 시연용 샘플 10월 한 달치 생성)
SAMPLE = [
    ('2026-10-01', '오늘부터 10월이다. 날씨가 선선해져서 산책하기 좋았다. 저녁에 친구랑 맛있는 걸 먹어서 기분이 좋았다.'),
    ('2026-10-02', '수업이 많아서 조금 피곤했다. 그래도 과제를 미리 끝내서 마음이 편하다.'),
    ('2026-10-03', '공휴일이라 늦잠을 잤다. 하루 종일 집에만 있었는데 좀 지루했다.'),
    ('2026-10-05', '알바에 손님이 너무 많아서 정신이 없었다. 실수까지 해서 혼나서 속상했다.'),
    ('2026-10-06', '동아리 사람들을 오랜만에 만났다. 다들 반갑게 맞아줘서 고마웠다. 이야기를 많이 해서 즐거웠다.'),
    ('2026-10-07', '중간고사 범위가 생각보다 넓어서 걱정된다. 계획을 세웠는데 지킬 수 있을지 모르겠다.'),
    ('2026-10-08', '팀 프로젝트 주제가 정해졌다. 재미있을 것 같아서 기대된다.'),
    ('2026-10-10', '주말인데 하루 종일 비가 왔다. 밖에 못 나가서 답답했다.'),
    ('2026-10-12', '도서관에서 하루 종일 공부했다. 집중이 잘 돼서 뿌듯했다.'),
    ('2026-10-13', '과제 마감이 겹쳐서 너무 힘들다. 잠을 거의 못 잤다.'),
    ('2026-10-14', '시험 공부를 하는데 이해가 안 되는 부분이 많아서 짜증났다. 내가 왜 이것도 못하나 싶어서 한심했다.'),
    ('2026-10-15', '친구가 모르는 부분을 친절하게 설명해줬다. 덕분에 이해가 돼서 정말 고마웠다.'),
    ('2026-10-16', '내일이 시험인데 너무 불안하다. 밤새 공부해야 할 것 같다.'),
    ('2026-10-17', '첫 시험을 봤는데 생각보다 어려웠다. 아는 문제도 실수해서 아쉽다.'),
    ('2026-10-19', '또 시험이다. 계속 공부만 하니까 지긋지긋하다.'),
    ('2026-10-20', '오늘 본 시험에서 공부한 부분이 많이 나왔다. 잘 본 것 같아서 너무 기쁘다!'),
    ('2026-10-21', '시험이 하나 남았는데 집중이 안 된다. 그냥 다 귀찮다.'),
    ('2026-10-23', '드디어 시험이 끝났다! 친구들이랑 고기 먹으러 갔는데 너무 행복했다.'),
    ('2026-10-24', '오랜만에 늦잠을 자고 영화를 봤다. 편하게 쉬어서 좋았다.'),
    ('2026-10-26', '시험 성적이 나왔는데 생각보다 낮아서 실망했다. 다음엔 더 열심히 해야겠다.'),
    ('2026-10-27', '팀 프로젝트 코드에서 계속 오류가 나서 답답했다. 그래도 원인을 찾아서 고치니까 뿌듯했다.'),
    ('2026-10-28', '알바 끝나고 집에 오니 너무 피곤하다. 아무것도 하기 싫다.'),
    ('2026-10-30', '축제에서 공연을 봤다. 분위기가 정말 신나고 재미있었다.'),
    ('2026-10-31', '이번 달은 정신없이 지나갔다. 힘들었지만 그래도 잘 버틴 것 같아서 다행이다.'),
]

if not DIARY_FILE.exists():
    pd.DataFrame(SAMPLE, columns=["date", "text"]).to_csv(DIARY_FILE, index=False, encoding="utf-8-sig")
    print("샘플 일기 생성:", DIARY_FILE)

diary = pd.read_csv(DIARY_FILE, encoding="utf-8-sig")
diary["date"] = pd.to_datetime(diary["date"]).dt.date
diary = diary.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)
print("일기 수:", len(diary), "|", diary["date"].min(), "~", diary["date"].max())
diary.head()

In [ ]:
# 8. 전체 분석 → 날짜별 저장
def daily_comment(r):
    v, g = r["valence"], r["main_group"]
    if v >= 0.3:
        return f"좋은 하루였네요! '{r['top_emotions'].split(', ')[0]}' 감정이 컸어요."
    if v <= -0.3:
        return {"피로·지루함": "많이 지친 하루였어요. 오늘은 푹 쉬어요.",
                "불안·당황":  "걱정이 많은 하루였네요. 너무 혼자 끌어안지 마세요.",
                "분노·불만":  "속상하고 화나는 일이 있었네요.",
                "슬픔·실망":  "마음이 무거운 하루였어요. 내일은 더 나아질 거예요."}.get(g, "힘든 하루였어요.")
    if g == "없음":
        return "잔잔하고 평범한 하루였어요."
    pos = any(r[f"r_{x}"] > 0 for x in POSITIVE); neg = any(r[f"r_{x}"] > 0 for x in NEGATIVE)
    if pos and neg:
        return "좋은 일과 힘든 일이 섞인 하루였어요."
    if g in NEGATIVE:
        return f"조금 '{g}' 쪽 감정이 있었던 하루예요."
    return "무난하게 괜찮은 하루였어요."

day_rows, sent_rows = [], []
for _, row in diary.iterrows():
    d, srows, _ = analyze_text(row["text"])
    day_rows.append({"date": row["date"], "text": row["text"], **d})
    sent_rows += [{"date": row["date"], **s} for s in srows]

daily = pd.DataFrame(day_rows)
daily["comment"] = daily.apply(daily_comment, axis=1)
sentences = pd.DataFrame(sent_rows)

daily.to_csv(output_dir / "diary_daily.csv", index=False, encoding="utf-8-sig")
sentences.to_csv(output_dir / "diary_sentences.csv", index=False, encoding="utf-8-sig")
print("저장:", output_dir / "diary_daily.csv", "/", output_dir / "diary_sentences.csv")
daily[["date", "n_sent", "main_group", "top_emotions", "valence", "comment"]]

In [ ]:
# 9. 주간 그래프 (기본: 마지막 일기가 있는 주, 월~일)
WEEK_OF = None    # 예: date(2026, 10, 19) 로 바꾸면 그 주

def plot_week(week_of=None):
    week_of = week_of or daily["date"].max()
    start = week_of - timedelta(days=week_of.weekday())
    days = [start + timedelta(days=i) for i in range(7)]
    w = daily.set_index("date").reindex(days)
    labels = [f"{d.month}/{d.day}\n{'월화수목금토일'[i]}" for i, d in enumerate(days)]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={"height_ratios": [1, 1.3]})
    x = np.arange(7)
    ax1.axhline(0, color="#999", lw=1)
    ax1.plot(x, w["valence"], "-", color="#888", lw=1.5)
    for i, (v, g) in enumerate(zip(w["valence"], w["main_group"])):
        if pd.notna(v):
            ax1.scatter(i, v, s=180, color=GROUP_COLORS[g], zorder=3, edgecolor="white", lw=1.5)
            ax1.annotate(g, (i, v), textcoords="offset points", xytext=(0, 12), ha="center", fontsize=8)
    ax1.set_ylim(-1.1, 1.1); ax1.set_xticks(x); ax1.set_xticklabels(labels)
    ax1.set_ylabel("긍정·부정 점수"); ax1.set_title(f"주간 감정 ({days[0]} ~ {days[-1]})", fontsize=13)

    groups = [g for g in GROUP_NAMES if g != "없음"]
    mat = np.array([[w.loc[d, f"r_{g}"] if pd.notna(w.loc[d, "valence"]) else np.nan for d in days] for g in groups], dtype=float)
    im = ax2.imshow(np.nan_to_num(mat, nan=0), cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
    for (i, j), v in np.ndenumerate(mat):
        ax2.text(j, i, "" if np.isnan(v) else f"{v:.0%}", ha="center", va="center", fontsize=8,
                 color="white" if v > 0.6 else "black")
    ax2.set_yticks(range(len(groups))); ax2.set_yticklabels(groups)
    ax2.set_xticks(x); ax2.set_xticklabels(labels)
    ax2.set_title("감정 그룹 비율 (그 감정이 나온 문장 비율)", fontsize=11)
    fig.colorbar(im, ax=ax2, fraction=0.025)
    plt.tight_layout()
    fig.savefig(output_dir / f"fig_week_{start}.png", dpi=150)
    plt.show()
    return w

_ = plot_week(WEEK_OF)

In [ ]:
# 10. 월간 그래프
MONTH = None    # 예: (2026, 10). None 이면 마지막 일기가 있는 달

def plot_month(month=None):
    y, m = month or (daily["date"].max().year, daily["date"].max().month)
    md_ = daily[[d.year == y and d.month == m for d in daily["date"]]].copy()
    md_["day"] = [d.day for d in md_["date"]]
    md_["week"] = [(d.day + date(y, m, 1).weekday() - 1) // 7 + 1 for d in md_["date"]]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [2, 1]})
    ax = axes[0]
    ax.axhline(0, color="#999", lw=1)
    ax.plot(md_["day"], md_["valence"], color="#888", lw=1.2)
    ax.scatter(md_["day"], md_["valence"], c=[GROUP_COLORS[g] for g in md_["main_group"]], s=70, zorder=3)
    wk = md_.groupby("week")["valence"].mean()
    for wnum, v in wk.items():
        d0 = max(1, (wnum - 1) * 7 - date(y, m, 1).weekday() + 1)
        d1 = min(calendar.monthrange(y, m)[1], d0 + 6 - (date(y, m, d0).weekday()))
        ax.hlines(v, d0, d1, colors="#333", linestyles="--", lw=1)
    ax.set_xlim(0.5, calendar.monthrange(y, m)[1] + 0.5); ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel("일"); ax.set_ylabel("긍정·부정 점수")
    ax.set_title(f"{y}년 {m}월 감정 흐름 (점선: 주간 평균)", fontsize=13)

    cnt = md_["main_group"].value_counts().reindex(GROUP_NAMES).fillna(0)
    cnt = cnt[cnt > 0]
    axes[1].barh(cnt.index[::-1], cnt.values[::-1], color=[GROUP_COLORS[g] for g in cnt.index[::-1]])
    for i, v in enumerate(cnt.values[::-1]):
        axes[1].text(v + 0.1, i, f"{int(v)}일", va="center")
    axes[1].set_title("대표 감정별 일수", fontsize=13)
    axes[1].set_xlim(0, cnt.max() + 1.5)

    handles = [Patch(color=GROUP_COLORS[g], label=g) for g in GROUP_NAMES if g in set(md_["main_group"])]
    ax.legend(handles=handles, loc="lower left", fontsize=8, ncol=3)
    plt.tight_layout()
    fig.savefig(output_dir / f"fig_month_{y}_{m:02d}.png", dpi=150)
    plt.show()
    return md_

_ = plot_month(MONTH)

In [ ]:
# 11. 감정 캘린더
def plot_calendar(month=None):
    y, m = month or (daily["date"].max().year, daily["date"].max().month)
    by_date = daily.set_index("date")
    weeks = calendar.Calendar(firstweekday=0).monthdatescalendar(y, m)

    fig, ax = plt.subplots(figsize=(10, 1.5 + 1.25 * len(weeks)))
    for c, name in enumerate("월화수목금토일"):
        ax.text(c + 0.5, -0.3, name, ha="center", va="center", fontsize=11,
                color="#D33" if c == 6 else ("#36C" if c == 5 else "#333"))
    for r, week in enumerate(weeks):
        for c, d in enumerate(week):
            inside = d.month == m
            has = inside and d in by_date.index
            g = by_date.loc[d, "main_group"] if has else None
            ax.add_patch(Rectangle((c + 0.04, r + 0.04), 0.92, 0.92,
                                   facecolor=GROUP_COLORS[g] if has else ("#F5F5F5" if inside else "white"),
                                   alpha=0.85 if has else 1, edgecolor="#DDD"))
            if inside:
                ax.text(c + 0.1, r + 0.2, str(d.day), fontsize=10, va="center",
                        color="#222" if has else "#AAA", fontweight="bold" if has else "normal")
            if has:
                ax.text(c + 0.5, r + 0.55, g, ha="center", va="center", fontsize=8.5)
                v = by_date.loc[d, "valence"]
                ax.text(c + 0.5, r + 0.8, f"{v:+.2f}", ha="center", va="center", fontsize=7.5, color="#333")
    ax.set_xlim(0, 7); ax.set_ylim(len(weeks), -0.6); ax.axis("off")
    ax.set_title(f"{y}년 {m}월 감정 캘린더", fontsize=15, pad=12)
    handles = [Patch(color=GROUP_COLORS[g], label=g) for g in GROUP_NAMES]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=7, fontsize=8, frameon=False)
    plt.tight_layout()
    fig.savefig(output_dir / f"fig_calendar_{y}_{m:02d}.png", dpi=150)
    plt.show()

plot_calendar(MONTH)

In [ ]:
# 12. 주간 / 월간 코멘트
def summary_comment(df, title):
    if df.empty:
        return f"[{title}] 일기가 없어요."
    groups = [g for g in GROUP_NAMES if g != "없음"]
    ratio = {g: df[f"r_{g}"].mean() for g in groups}
    top_g = max(ratio, key=ratio.get)
    best, worst = df.loc[df["valence"].idxmax()], df.loc[df["valence"].idxmin()]
    avg = df["valence"].mean()
    mood = "전반적으로 긍정적인" if avg >= 0.2 else ("전반적으로 힘든" if avg <= -0.2 else "좋고 힘든 일이 섞인")
    lines = [f"[{title}] 일기 {len(df)}일 · {mood} 시기였어요 (평균 점수 {avg:+.2f}).",
             f"- 가장 많이 나온 감정 그룹: {top_g} (문장의 {ratio[top_g]:.0%})",
             f"- 가장 좋았던 날: {best['date']} ({best['top_emotions']})",
             f"- 가장 힘들었던 날: {worst['date']} ({worst['top_emotions']})"]
    if len(df) >= 4:
        half = len(df) // 2
        diff = df["valence"].iloc[half:].mean() - df["valence"].iloc[:half].mean()
        lines.append("- 후반으로 갈수록 " + ("기분이 좋아졌어요." if diff > 0.15 else
                                          "기분이 가라앉았어요. 쉬어 가도 괜찮아요." if diff < -0.15 else "비슷하게 유지됐어요."))
    return "\n".join(lines)

last = daily["date"].max()
wstart = last - timedelta(days=last.weekday())
week_df = daily[(daily["date"] >= wstart) & (daily["date"] <= wstart + timedelta(days=6))].reset_index(drop=True)
month_df = daily[[d.year == last.year and d.month == last.month for d in daily["date"]]].reset_index(drop=True)
print(summary_comment(week_df, f"이번 주 {wstart}~"))
print()
print(summary_comment(month_df, f"{last.year}년 {last.month}월"))

In [ ]:
# 13. 오늘 일기 쓰기 → 바로 분석 + 파일에 추가 저장
def write_diary(text, day=None, save=True):
    day = day or date.today()
    d, srows, _ = analyze_text(text)
    r = {"date": day, "text": text, **d}
    r["comment"] = daily_comment(r)
    print(f"[{day}]")
    for s in srows:
        print(f"- {s['sentence']}\n    → {s['emotions']}")
    print(f"\n대표 감정: {d['main_group']} | 주요 감정: {d['top_emotions']} | 점수 {d['valence']:+.2f}")
    print("코멘트:", r["comment"])
    if save:
        old = pd.read_csv(DIARY_FILE, encoding="utf-8-sig")
        old = old[old["date"] != str(day)]
        pd.concat([old, pd.DataFrame([{"date": str(day), "text": text}])]) \
          .sort_values("date").to_csv(DIARY_FILE, index=False, encoding="utf-8-sig")
        print("저장됨 → 위의 셀 7부터 다시 실행하면 그래프·캘린더에 반영")
    return r

_ = write_diary("오늘 시험을 잘 봐서 뿌듯했지만 다음 발표가 걱정되고 너무 피곤하다.", save=False)   # save=True 로 바꾸면 저장